In [ ]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
from ipywidgets import interactive, Dropdown, FloatSlider, HBox, VBox, HTML, Layout
from IPython.display import display
import textwrap

# ============================================================
# WINDOW SELECTION FROM FIR DESIGN SPECIFICATIONS
# ============================================================

WINDOW_DATA = {
    'Rectangular': {'dw':4.0, 'dwm':1.8, 'As':21.0},
    'Bartlett': {'dw':8.0, 'dwm':6.1, 'As':25.0},
    'von Hann': {'dw':8.0, 'dwm':6.2, 'As':44.0},
    'Hamming': {'dw':8.0, 'dwm':6.6, 'As':53.0},
    'Blackman': {'dw':12.0, 'dwm':11.0, 'As':74.0}
}

WINDOWS = list(WINDOW_DATA.keys())

# ============================================================
# CSS
# ============================================================

style_html = HTML("""
<style>
.ws-root {width:970px; max-width:970px; font-family:Arial,sans-serif;}
.ws-header {background:linear-gradient(90deg,#5b2c83,#8e44ad); color:white; padding:10px 15px; border-radius:8px 8px 0 0; font-size:19px; font-weight:bold;}
.ws-intro {background:#faf6fc; border:1px solid #d7c4e2; border-top:none; padding:8px 12px; border-radius:0 0 8px 8px; font-size:12px; line-height:1.48; margin-bottom:8px;}
.ws-accent {font-weight:bold; color:#6c3483;}
.ws-title {font-size:12.5px; font-weight:bold; margin:0 0 5px 2px; color:#6c3483;}
.jupyter-widgets-output-area, .widget-output, .output_area, .output_subarea {overflow-x:visible !important; max-width:none !important;}
</style>
""")

# ============================================================
# DOCUMENTATION
# ============================================================

header_html = HTML("""
<div class="ws-root">
<div class="ws-header">Window Selection from FIR Design Specifications</div>
<div class="ws-intro">

<span class="ws-accent">Automatic mode:</span>
the user specifies the FIR requirements and the notebook automatically selects the first tabulated window whose minimum stopband attenuation
<b>A<sub>s</sub><sup>min</sup></b> is at least equal to the required stopband attenuation <b>A<sub>s</sub></b>.
The manual window menu is disabled because the choice is made automatically.
<br><br>

<span class="ws-accent">Manual mode:</span>
the user may select any window explicitly. The automatic recommendation remains visible as a reference, while the manually selected window is independently classified as PASS or FAIL.
<br><br>

<span class="ws-accent">Design specifications:</span>
the user enters the passband ripple <b>A<sub>p</sub></b> and the required stopband attenuation <b>A<sub>s</sub></b> directly in dB.
The theoretical table contains <b>A<sub>s</sub><sup>min</sup></b>, so the automatic choice is based directly on
<b>A<sub>s</sub><sup>min</sup> ≥ A<sub>s</sub></b>.
The value A<sub>p</sub> remains part of the filter specifications but is not used by this particular table to select the window.
<br><br>

<span class="ws-accent">Required FIR length:</span>
for normalized transition width Δω/π, the tabulated relation
<b>Δω = Cπ/N</b>
gives
<b>N = ceil[C/(Δω/π)]</b>.
For band-pass and band-stop filters two transition widths exist and the narrower one is used.
<br><br>

<span class="ws-accent">Window Suitability Score:</span>
this is <b>not a standard DSP quantity</b>. It is a pedagogical comparison index defined specifically for this notebook.
The automatically recommended window is used as the reference design.
The complete equations used to calculate the score are displayed in the output below the graphs.
<br><br>

<span class="ws-accent">Important:</span>
the Suitability Score does not determine PASS or FAIL.
The theoretical decision depends only on whether
<b>A<sub>s</sub><sup>min</sup> ≥ A<sub>s</sub></b>.
<br><br>

<span class="ws-accent">Frequency controls:</span>
the sliders constrain one another automatically.
For band-pass filters:
<b>ωs1 &lt; ωp1 &lt; ωp2 &lt; ωs2</b>.
For band-stop filters:
<b>ωp1 &lt; ωs1 &lt; ωs2 &lt; ωp2</b>.

</div>
</div>
""")

# ============================================================
# DESIGN CALCULATIONS
# ============================================================

def calculate_required_length(window_name, transition_width):
    if transition_width <= 1e-12: return None
    return max(2, int(np.ceil(WINDOW_DATA[window_name]['dw'] / transition_width)))

def automatic_window(As_req):
    for name in WINDOWS:
        if WINDOW_DATA[name]['As'] >= As_req: return name
    return None

def calculate_scores(transition_width, reference_window, As_req):
    lengths = {name:calculate_required_length(name, transition_width) for name in WINDOWS}

    if reference_window is None:
        return lengths, {name:np.nan for name in WINDOWS}, {name:np.nan for name in WINDOWS}

    ref = WINDOW_DATA[reference_window]
    N_ref = lengths[reference_window]
    deviations, scores = {}, {}

    for name in WINDOWS:
        data = WINDOW_DATA[name]
        dA = (data['As'] - ref['As']) / max(As_req,1e-12)
        dN = (lengths[name] - N_ref) / 100.0
        dM = (data['dwm'] - ref['dwm']) / 10.0
        D = np.sqrt(dA**2 + dN**2 + dM**2)
        deviations[name] = D
        scores[name] = 100.0 / (1.0 + D)

    return lengths, deviations, scores

# ============================================================
# FREQUENCY SPECIFICATIONS
# ============================================================

def interpret_frequencies(filter_type, f1, f2, f3, f4):
    if filter_type == 'Low-pass':
        return {'wp':f1,'ws':f2,'wc':0.5*(f1+f2),'dw1':f2-f1,'dw2':None,'dw':f2-f1}

    if filter_type == 'High-pass':
        return {'ws':f1,'wp':f2,'wc':0.5*(f1+f2),'dw1':f2-f1,'dw2':None,'dw':f2-f1}

    if filter_type == 'Band-pass':
        dw1, dw2 = f2-f1, f4-f3
        return {'ws1':f1,'wp1':f2,'wp2':f3,'ws2':f4,'wc1':0.5*(f1+f2),'wc2':0.5*(f3+f4),'dw1':dw1,'dw2':dw2,'dw':min(dw1,dw2)}

    dw1, dw2 = f2-f1, f4-f3
    return {'wp1':f1,'ws1':f2,'ws2':f3,'wp2':f4,'wc1':0.5*(f1+f2),'wc2':0.5*(f3+f4),'dw1':dw1,'dw2':dw2,'dw':min(dw1,dw2)}

def ideal_response(filter_type, omega, specs):
    if filter_type == 'Low-pass': return np.where(omega <= specs['wc'],1.0,0.0)
    if filter_type == 'High-pass': return np.where(omega >= specs['wc'],1.0,0.0)
    if filter_type == 'Band-pass': return np.where((omega >= specs['wc1']) & (omega <= specs['wc2']),1.0,0.0)
    return np.where((omega <= specs['wc1']) | (omega >= specs['wc2']),1.0,0.0)

# ============================================================
# DASHBOARD CARD
# ============================================================

def draw_card(ax, x, y, w, h, title, rows, title_color='#6c3483'):
    box = FancyBboxPatch((x,y-h),w,h,boxstyle='round,pad=0.010,rounding_size=0.018',transform=ax.transAxes,facecolor='white',edgecolor='#c9bfd0',linewidth=1.0)
    ax.add_patch(box)

    header_h = min(0.065,0.24*h)
    header = FancyBboxPatch((x,y-header_h),w,header_h,boxstyle='round,pad=0.010,rounding_size=0.018',transform=ax.transAxes,facecolor=title_color,edgecolor=title_color,linewidth=0)
    ax.add_patch(header)

    ax.text(x+0.04*w,y-header_h/2,title,transform=ax.transAxes,ha='left',va='center',color='white',fontsize=10.2,fontweight='bold')

    top = y-header_h-0.035
    bottom = y-h+0.035
    positions = np.linspace(top,bottom,len(rows))

    for yy,row in zip(positions,rows):
        label,value = row[0],row[1]
        value_color = row[2] if len(row) > 2 else '#202020'
        ax.text(x+0.05*w,yy,label,transform=ax.transAxes,ha='left',va='center',fontsize=9.5)
        ax.text(x+0.95*w,yy,value,transform=ax.transAxes,ha='right',va='center',fontsize=9.8,fontweight='bold',color=value_color)

# ============================================================
# MAIN INTERACTIVE FUNCTION
# ============================================================

def plot_window_selection(filter_type='Low-pass', selection_mode='Automatic', manual_window='von Hann', f1=0.20, f2=0.30, f3=0.65, f4=0.75, Ap=1.0, As=40.0):

    specs = interpret_frequencies(filter_type,f1,f2,f3,f4)
    transition_width = specs['dw']

    auto_window = automatic_window(As)
    selected_window = auto_window if selection_mode == 'Automatic' else manual_window

    lengths,deviations,scores = calculate_scores(transition_width,auto_window,As)
    status = {name:WINDOW_DATA[name]['As'] >= As for name in WINDOWS}

    selected_N = lengths[selected_window] if selected_window is not None else None
    selected_status = status[selected_window] if selected_window is not None else False

    # ========================================================
    # FIGURE
    # ========================================================

    fig = plt.figure(figsize=(16.5,10.4))

    grid = fig.add_gridspec(
        3,3,
        width_ratios=[1.10,1.82,1.05],
        height_ratios=[1.0,1.0,0.62],
        wspace=0.36,
        hspace=0.52
    )

    ax_spec = fig.add_subplot(grid[0,0])
    ax_score = fig.add_subplot(grid[1,0])
    ax_table = fig.add_subplot(grid[0:2,1])
    ax_info = fig.add_subplot(grid[0:2,2])
    ax_formula = fig.add_subplot(grid[2,:])

    # ========================================================
    # DESIGN SPECIFICATIONS
    # ========================================================

    omega = np.linspace(0.0,1.0,1600)
    ideal = ideal_response(filter_type,omega,specs)

    ax_spec.plot(omega,ideal,linewidth=1.9)

    if filter_type == 'Low-pass':
        ax_spec.axvspan(specs['wp'],specs['ws'],alpha=0.12)
        ax_spec.axvline(specs['wp'],linestyle='--',linewidth=1.0,label=r'$\omega_p$')
        ax_spec.axvline(specs['ws'],linestyle='--',linewidth=1.0,label=r'$\omega_s$')
        ax_spec.axvline(specs['wc'],linestyle=':',linewidth=1.3,label=r'$\omega_c$')

    elif filter_type == 'High-pass':
        ax_spec.axvspan(specs['ws'],specs['wp'],alpha=0.12)
        ax_spec.axvline(specs['ws'],linestyle='--',linewidth=1.0,label=r'$\omega_s$')
        ax_spec.axvline(specs['wp'],linestyle='--',linewidth=1.0,label=r'$\omega_p$')
        ax_spec.axvline(specs['wc'],linestyle=':',linewidth=1.3,label=r'$\omega_c$')

    elif filter_type == 'Band-pass':
        ax_spec.axvspan(specs['ws1'],specs['wp1'],alpha=0.12)
        ax_spec.axvspan(specs['wp2'],specs['ws2'],alpha=0.12)
        ax_spec.axvline(specs['ws1'],linestyle='--',linewidth=0.9,label=r'$\omega_{s1}$')
        ax_spec.axvline(specs['wp1'],linestyle='--',linewidth=0.9,label=r'$\omega_{p1}$')
        ax_spec.axvline(specs['wp2'],linestyle='--',linewidth=0.9,label=r'$\omega_{p2}$')
        ax_spec.axvline(specs['ws2'],linestyle='--',linewidth=0.9,label=r'$\omega_{s2}$')

    else:
        ax_spec.axvspan(specs['wp1'],specs['ws1'],alpha=0.12)
        ax_spec.axvspan(specs['ws2'],specs['wp2'],alpha=0.12)
        ax_spec.axvline(specs['wp1'],linestyle='--',linewidth=0.9,label=r'$\omega_{p1}$')
        ax_spec.axvline(specs['ws1'],linestyle='--',linewidth=0.9,label=r'$\omega_{s1}$')
        ax_spec.axvline(specs['ws2'],linestyle='--',linewidth=0.9,label=r'$\omega_{s2}$')
        ax_spec.axvline(specs['wp2'],linestyle='--',linewidth=0.9,label=r'$\omega_{p2}$')

    ax_spec.set_xlim(0.0,1.0)
    ax_spec.set_ylim(-0.08,1.15)
    ax_spec.set_xticks([0.0,0.25,0.50,0.75,1.0])
    ax_spec.set_xticklabels(['0',r'$0.25\pi$',r'$0.5\pi$',r'$0.75\pi$',r'$\pi$'])
    ax_spec.set_xlabel(r'Frequency $\omega$')
    ax_spec.set_ylabel('Ideal magnitude')
    ax_spec.set_title('Design Specifications')
    ax_spec.grid(True,linestyle=':',alpha=0.25)
    ax_spec.legend(loc='upper center',bbox_to_anchor=(0.5,-0.20),ncol=2 if filter_type in ['Band-pass','Band-stop'] else 3,frameon=False,fontsize=8.8)

    # ========================================================
    # WINDOW SUITABILITY SCORE
    # ========================================================

    y = np.arange(len(WINDOWS))
    score_values = np.array([scores[name] if np.isfinite(scores[name]) else 0.0 for name in WINDOWS])

    ax_score.barh(y,score_values)
    ax_score.set_yticks(y)
    ax_score.set_yticklabels(WINDOWS)
    ax_score.invert_yaxis()
    ax_score.set_xlim(0,115)
    ax_score.set_xticks([0,20,40,60,80,100])
    ax_score.set_xlabel('Suitability Score')
    ax_score.set_title('Window Suitability Score')
    ax_score.grid(True,axis='x',linestyle=':',alpha=0.25)

    for i,name in enumerate(WINDOWS):
        ax_score.text(score_values[i]+2.0,i,f'{score_values[i]:.1f}',va='center',ha='left',fontsize=9.2)

    # ========================================================
    # WINDOW COMPARISON TABLE
    # ========================================================

    ax_table.axis('off')
    rows = []

    for name in WINDOWS:
        data = WINDOW_DATA[name]
        marker = '★ ' if name == auto_window else ''
        N_text = str(lengths[name]) if lengths[name] is not None else '—'
        score_text = f'{scores[name]:.1f}' if np.isfinite(scores[name]) else '—'
        rows.append([marker+name,f'{data["As"]:.0f}',f'{data["dw"]:.1f}π/N',f'{data["dwm"]:.1f}π/N',N_text,'PASS' if status[name] else 'FAIL',score_text])

    table = ax_table.table(
        cellText=rows,
        colLabels=['Window',r'$A_s^{min}$',r'$\Delta\omega$',r'$\Delta\omega_m$','N','Status','Score'],
        cellLoc='center',
        colLoc='center',
        colWidths=[0.25,0.15,0.17,0.18,0.13,0.15,0.14],
        bbox=[0.00,0.43,1.00,0.52]
    )

    table.auto_set_font_size(False)
    table.set_fontsize(9.3)
    table.scale(1.0,1.42)

    for j in range(7):
        table[(0,j)].set_text_props(weight='bold')
        table[(0,j)].set_facecolor('#eadcf1')

    for i,name in enumerate(WINDOWS,start=1):
        if name == auto_window:
            table[(i,0)].set_text_props(weight='bold')

        table[(i,5)].set_facecolor('#e6f4ea' if status[name] else '#fce8e6')
        table[(i,5)].set_text_props(weight='bold')

    ax_table.text(0.00,0.375,'★ Automatically recommended window',transform=ax_table.transAxes,fontsize=9.8,fontweight='bold')

    if auto_window is not None:
        explanation = f'Required stopband attenuation: {As:.2f} dB. {auto_window} is the first window in the theoretical table whose minimum stopband attenuation satisfies this requirement. Windows below it fail the attenuation specification, while stronger windows remain valid alternatives but generally involve a different transition-width / filter-length trade-off.'
    else:
        explanation = f'Required stopband attenuation: {As:.2f} dB. This exceeds the 74 dB capability of all windows included in the theoretical table. None of the listed windows satisfies the specification.'

    ax_table.text(0.00,0.315,'\n'.join(textwrap.wrap(explanation,width=82)),transform=ax_table.transAxes,ha='left',va='top',fontsize=9.5,linespacing=1.35)

    # ========================================================
    # INFORMATION DASHBOARD
    # ========================================================

    ax_info.axis('off')

    if filter_type in ['Low-pass','High-pass']:
        specification_rows = [
            ('Filter type',filter_type),
            (r'$\omega_p$',f'{specs["wp"]:.2f}π'),
            (r'$\omega_s$',f'{specs["ws"]:.2f}π'),
            (r'$\omega_c$',f'{specs["wc"]:.2f}π'),
            (r'$\Delta\omega$',f'{specs["dw"]:.2f}π'),
            (r'$A_p$',f'{Ap:.2f} dB'),
            (r'$A_s$',f'{As:.2f} dB')
        ]

    else:
        specification_rows = [
            ('Filter type',filter_type),
            (r'$\Delta\omega_1$',f'{specs["dw1"]:.2f}π'),
            (r'$\Delta\omega_2$',f'{specs["dw2"]:.2f}π'),
            (r'Governing $\Delta\omega$',f'{specs["dw"]:.2f}π'),
            (r'$A_p$',f'{Ap:.2f} dB'),
            (r'$A_s$',f'{As:.2f} dB')
        ]

    if auto_window is not None:
        recommendation_rows = [
            ('Recommended',auto_window),
            ('Required N',str(lengths[auto_window])),
            ('Filter order',str(lengths[auto_window]-1))
        ]
    else:
        recommendation_rows = [
            ('Recommended','NONE'),
            ('Required N','—'),
            ('Filter order','—')
        ]

    selected_score = f'{scores[selected_window]:.1f}' if selected_window is not None and np.isfinite(scores[selected_window]) else '—'
    status_color = '#117864' if selected_status else '#a93226'

    current_rows = [
        ('Mode',selection_mode),
        ('Window',selected_window if selected_window is not None else 'NONE'),
        ('Required N',str(selected_N) if selected_N is not None else '—'),
        ('Score',selected_score),
        ('Status','PASS' if selected_status else 'FAIL',status_color)
    ]

    draw_card(ax_info,0.02,0.98,0.96,0.40,'DESIGN SPECIFICATIONS',specification_rows,'#5b2c83')
    draw_card(ax_info,0.02,0.53,0.96,0.20,'AUTOMATIC RECOMMENDATION',recommendation_rows,'#2874a6')
    draw_card(ax_info,0.02,0.28,0.96,0.23,'CURRENT SELECTION',current_rows,'#117864' if selected_status else '#a93226')

    # ========================================================
    # LARGE, READABLE SCORE EQUATION PANEL
    # ========================================================

    ax_formula.axis('off')

    outer_box = FancyBboxPatch((0.015,0.04),0.97,0.92,boxstyle='round,pad=0.015,rounding_size=0.020',transform=ax_formula.transAxes,facecolor='#fbf8fd',edgecolor='#cdb7d8',linewidth=1.0)
    ax_formula.add_patch(outer_box)

    ax_formula.text(0.035,0.88,'WINDOW SUITABILITY SCORE',transform=ax_formula.transAxes,ha='left',va='center',fontsize=13.0,fontweight='bold',color='#6c3483')

    left_box = FancyBboxPatch((0.035,0.31),0.59,0.43,boxstyle='round,pad=0.012,rounding_size=0.015',transform=ax_formula.transAxes,facecolor='white',edgecolor='#ddd2e3',linewidth=1.0)
    right_box = FancyBboxPatch((0.655,0.31),0.31,0.43,boxstyle='round,pad=0.012,rounding_size=0.015',transform=ax_formula.transAxes,facecolor='white',edgecolor='#ddd2e3',linewidth=1.0)

    ax_formula.add_patch(left_box)
    ax_formula.add_patch(right_box)

    ax_formula.text(
        0.330,
        0.525,
        r'$D_i=\sqrt{\left(\frac{A_{s,i}-A_{s,*}}{A_s^{req}}\right)^2+\left(\frac{N_i-N_*}{100}\right)^2+\left(\frac{C_{m,i}-C_{m,*}}{10}\right)^2}$',
        transform=ax_formula.transAxes,
        ha='center',
        va='center',
        fontsize=18.0
    )

    ax_formula.text(
        0.810,
        0.525,
        r'$S_i=\frac{100}{1+D_i}$',
        transform=ax_formula.transAxes,
        ha='center',
        va='center',
        fontsize=22.0
    )

    ax_formula.text(
        0.50,
        0.16,
        r'$*=$ automatically recommended reference window     $\qquad$     $\Delta\omega_m=C_m\pi/N$     $\qquad$     $D_*=0,\;S_*=100$',
        transform=ax_formula.transAxes,
        ha='center',
        va='center',
        fontsize=11.5
    )

    fig.suptitle('FIR Window Selection from Design Specifications',fontsize=13.5)
    plt.subplots_adjust(left=0.045,right=0.99,top=0.92,bottom=0.055)

    plt.show()
    plt.close(fig)

# ============================================================
# CONTROLS
# ============================================================

filter_selector = Dropdown(options=['Low-pass','High-pass','Band-pass','Band-stop'],value='Low-pass',description='Filter:',style={'description_width':'45px'},layout=Layout(width='210px'))
mode_selector = Dropdown(options=['Automatic','Manual'],value='Automatic',description='Mode:',style={'description_width':'42px'},layout=Layout(width='210px'))
manual_window_selector = Dropdown(options=WINDOWS,value='von Hann',description='Window:',style={'description_width':'52px'},layout=Layout(width='235px'))

f1_slider = FloatSlider(value=0.20,min=0.02,max=0.95,step=0.01,description='ωp / π:',continuous_update=True,readout_format='.2f',style={'description_width':'55px'},layout=Layout(width='225px'))
f2_slider = FloatSlider(value=0.30,min=0.03,max=0.97,step=0.01,description='ωs / π:',continuous_update=True,readout_format='.2f',style={'description_width':'55px'},layout=Layout(width='225px'))
f3_slider = FloatSlider(value=0.65,min=0.03,max=0.97,step=0.01,description='ω3 / π:',continuous_update=True,readout_format='.2f',style={'description_width':'55px'},layout=Layout(width='225px'))
f4_slider = FloatSlider(value=0.75,min=0.04,max=0.98,step=0.01,description='ω4 / π:',continuous_update=True,readout_format='.2f',style={'description_width':'55px'},layout=Layout(width='225px'))

Ap_slider = FloatSlider(value=1.0,min=0.10,max=5.00,step=0.10,description='Ap [dB]:',continuous_update=True,readout_format='.1f',style={'description_width':'60px'},layout=Layout(width='300px'))
As_slider = FloatSlider(value=40.0,min=10.0,max=80.0,step=1.0,description='As [dB]:',continuous_update=True,readout_format='.0f',style={'description_width':'60px'},layout=Layout(width='300px'))

# ============================================================
# DYNAMIC FREQUENCY CONSTRAINTS
# ============================================================

def reset_frequency_bounds():
    f1_slider.min,f1_slider.max = 0.02,0.95
    f2_slider.min,f2_slider.max = 0.03,0.97
    f3_slider.min,f3_slider.max = 0.03,0.97
    f4_slider.min,f4_slider.max = 0.04,0.98

def update_frequency_bounds(change=None):
    gap = 0.01

    if filter_selector.value in ['Low-pass','High-pass']:
        f1_slider.max = max(f1_slider.min,f2_slider.value-gap)
        f2_slider.min = min(f2_slider.max,f1_slider.value+gap)

    else:
        f1_slider.max = max(f1_slider.min,f2_slider.value-gap)
        f2_slider.min = min(f2_slider.max,f1_slider.value+gap)
        f2_slider.max = max(f2_slider.min,f3_slider.value-gap)
        f3_slider.min = min(f3_slider.max,f2_slider.value+gap)
        f3_slider.max = max(f3_slider.min,f4_slider.value-gap)
        f4_slider.min = min(f4_slider.max,f3_slider.value+gap)

# ============================================================
# MODE / FILTER CONTROL LOGIC
# ============================================================

def update_mode_controls(change=None):
    manual_window_selector.disabled = mode_selector.value == 'Automatic'

def update_filter_controls(change=None):
    reset_frequency_bounds()

    if filter_selector.value == 'Low-pass':
        f1_slider.description,f2_slider.description,f3_slider.description,f4_slider.description = 'ωp / π:','ωs / π:','—','—'
        f1_slider.value,f2_slider.value,f3_slider.value,f4_slider.value = 0.20,0.30,0.65,0.75
        f3_slider.disabled,f4_slider.disabled = True,True

    elif filter_selector.value == 'High-pass':
        f1_slider.description,f2_slider.description,f3_slider.description,f4_slider.description = 'ωs / π:','ωp / π:','—','—'
        f1_slider.value,f2_slider.value,f3_slider.value,f4_slider.value = 0.20,0.30,0.65,0.75
        f3_slider.disabled,f4_slider.disabled = True,True

    elif filter_selector.value == 'Band-pass':
        f1_slider.description,f2_slider.description,f3_slider.description,f4_slider.description = 'ωs1 / π:','ωp1 / π:','ωp2 / π:','ωs2 / π:'
        f1_slider.value,f2_slider.value,f3_slider.value,f4_slider.value = 0.15,0.25,0.65,0.75
        f3_slider.disabled,f4_slider.disabled = False,False

    else:
        f1_slider.description,f2_slider.description,f3_slider.description,f4_slider.description = 'ωp1 / π:','ωs1 / π:','ωs2 / π:','ωp2 / π:'
        f1_slider.value,f2_slider.value,f3_slider.value,f4_slider.value = 0.15,0.25,0.65,0.75
        f3_slider.disabled,f4_slider.disabled = False,False

    update_frequency_bounds()

# ============================================================
# OBSERVERS
# ============================================================

mode_selector.observe(update_mode_controls,names='value')
filter_selector.observe(update_filter_controls,names='value')

f1_slider.observe(update_frequency_bounds,names='value')
f2_slider.observe(update_frequency_bounds,names='value')
f3_slider.observe(update_frequency_bounds,names='value')
f4_slider.observe(update_frequency_bounds,names='value')

update_mode_controls()
update_filter_controls()

# ============================================================
# INTERACTIVE OBJECT
# ============================================================

widget_plot = interactive(plot_window_selection,filter_type=filter_selector,selection_mode=mode_selector,manual_window=manual_window_selector,f1=f1_slider,f2=f2_slider,f3=f3_slider,f4=f4_slider,Ap=Ap_slider,As=As_slider)

plot_output = widget_plot.children[-1]
plot_output.layout = Layout(width='auto',overflow='visible')

# ============================================================
# CONTROL LAYOUT
# ============================================================

selection_row = HBox([filter_selector,mode_selector,manual_window_selector],layout=Layout(width='720px',justify_content='space-between',align_items='center'))
selection_box = VBox([HTML("<div class='ws-title'>Filter and Window Selection</div>"),selection_row],layout=Layout(width='970px',border='1px solid #d7c4e2',padding='7px 10px',overflow='visible'))

frequency_row = HBox([f1_slider,f2_slider,f3_slider,f4_slider],layout=Layout(width='940px',justify_content='space-between',align_items='center'))
attenuation_row = HBox([Ap_slider,As_slider],layout=Layout(width='620px',justify_content='space-between',align_items='center'))

specification_box = VBox([HTML("<div class='ws-title'>Operating Specifications</div>"),frequency_row,attenuation_row],layout=Layout(width='970px',border='1px solid #d7c4e2',padding='7px 10px',overflow='visible'))

main_layout = VBox([header_html,selection_box,specification_box,plot_output],layout=Layout(width='970px',overflow='visible',align_items='flex-start'))

display(style_html)
display(main_layout)